In [ ]:
from pathlib import Path
import astropy
import os, sys
import pandas as pd
import numpy as np
from astropy.table import Table, join, vstack
import desispec
import matplotlib.pyplot as plt

# from sklearn.linear_model import LinearRegression
from scipy.stats import binned_statistic
from matplotlib.gridspec import GridSpec
from utils import better_step

from astropy.coordinates import SkyCoord
import astropy.units as u

In [ ]:
# figure defaults for AASTEX AJ

COLUMN_WIDTH = 242.26653/72.27 #in inches
TEXT_WIDTH = 513.11743/72.27
SMALL_SIZE = 9 # in pts
NORMAL_SIZE = 10 
BIG_SIZE = 12
FONT_FAMILY = 'Nimbus Roman No9 L'



params = {
    "font.family": FONT_FAMILY,
    "font.size": NORMAL_SIZE,
    "axes.titlesize": NORMAL_SIZE,
    "axes.labelsize": NORMAL_SIZE,
    
    "xtick.labelsize": SMALL_SIZE,
    "ytick.labelsize": SMALL_SIZE,
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "legend.fontsize": NORMAL_SIZE,
    
    "figure.facecolor": "w",
    "figure.dpi": 300,
    'mathtext.fontset' : "cm"
    
}

plt.rcParams.update(params)

In [ ]:
data_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/MERGED")
cat = Table.read(data_path / "merged_cat_LSST_WL_Y1.fits")


In [ ]:
# 
field_names = ["XMMLSS","COSMOS"]
fig, ax = plt.subplots(1,2, figsize=(TEXT_WIDTH, 0.4*TEXT_WIDTH))

# fig = plt.figure( figsize=(TEXT_WIDTH,0.3*TEXT_WIDTH))
# gs = GridSpec(1, 12, figure=fig)
# ax = [fig.add_subplot(gs[0:3]), fig.add_subplot(gs[3:6]), fig.add_subplot(gs[6:9]), fig.add_subplot(gs[11]),]


for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    print(len(cat_plot))
    # ax[i].grid(zorder=0)
    s = ax[i].scatter(cat_plot["TARGET_RA"],cat_plot["TARGET_DEC"],marker=".",
               s=4,c=cat_plot["EXPTIME"]/60,vmin=15,vmax=500,cmap="viridis",rasterized=True)
    if field=="XMM":
        field="XMMLSS"
    ax[i].set_title("DESI-"+field,fontsize=BIG_SIZE)
ax[1].set_xlim(148.2, 152)
ax[1].set_ylim(0.4, 4.1)
# #HERCULES
# ax[2].set_xlim(243.7, 248.5,)
# ax[2].set_ylim( 41.5,45.1)



ax[0].set_xlabel("R.A. [deg]")
ax[1].set_xlabel("R.A. [deg]")
ax[0].set_ylabel("DEC. [deg]")
ax[1].set_ylabel("DEC. [deg]")

# fig.subplots_adjust(left = 0, right=0.01)
cbar_ax = fig.add_axes([0.95, 0.15, 0.02, 0.7])
cbar = fig.colorbar(s, fraction=0.46,cax=cbar_ax)
cbar.set_label("Exposure Time [min]",rotation=-90,labelpad = 15)
# cbar.ax.set_aspect(0.5)

plt.savefig("./figs/points_on_sky.pdf",bbox_inches="tight",dpi=300)

In [ ]:
cat["success"] = (cat["VI_quality"]>2)
cat = vstack([cat[~cat["success"]],
                      cat[cat["success"] & (cat["Z"]>0.001)]])


In [ ]:
cat_cosmos = cat[cat["FIELD_NAME"]=="COSMOS"]
cat_xmm = cat[cat["FIELD_NAME"]=="XMMLSS"]
cat_hercules = cat[cat["FIELD_NAME"]=="HERCULES"]
print(f"Total in COSMOS:{len(cat_cosmos)}, Failures: {(~cat_cosmos['success']).sum()}, Faint: {(cat_cosmos['mag_i']>=23).sum()}")
print(f"Total in XMM:{len(cat_xmm)}, Failures: {(~cat_xmm['success']).sum()}, Faint: {(cat_xmm['mag_i']>=23).sum()}")
print(f"Total in HERCULES:{len(cat_hercules)}, Failures: {(~cat_hercules['success']).sum()}, Faint: {(cat_hercules['mag_i']>=23).sum()}")

In [ ]:
from scipy.stats import gaussian_kde

# Magnitude histogram of failures split by field as a single stacked bar plot
fig, ax = plt.subplots(1, 2, figsize=(1.2*TEXT_WIDTH, 0.4*TEXT_WIDTH))
bins = np.linspace(22,24.5,10)
ax[0].hist(
    [cat_xmm["mag_i"][~cat_xmm["success"]], cat_cosmos["mag_i"][~cat_cosmos["success"]]],
    bins=bins,
    stacked=True,
    label=[ "DESI-XMM", "DESI-COSMOS"],
    color=["C0", "C1"],
    alpha=0.7,
    rwidth=0.8
)
ax[0].set_xlabel("i mag")
ax[0].set_ylabel("Number of redshift failures")
ax[0].legend(frameon=False, loc=(0.41,0.78))


# Prepare data for COSMOS failures
x_cosmos = cat_cosmos["mag_i"][~cat_cosmos["success"]]
y_cosmos = cat_cosmos["EXPTIME"][~cat_cosmos["success"]]/60

# Prepare data for XMM failures
x_xmm = cat_xmm["mag_i"][~cat_xmm["success"]]
y_xmm = cat_xmm["EXPTIME"][~cat_xmm["success"]]/60

# Create grid for contour
xi = np.linspace(bins.min(), bins.max(), 100)
yi = np.linspace(min(np.min(y_cosmos), np.min(y_xmm)), max(np.max(y_cosmos), np.max(y_xmm)), 100)
X, Y = np.meshgrid(xi, yi)

# COSMOS contour
kde_cosmos = gaussian_kde(np.vstack([x_cosmos, y_cosmos]))
Z_cosmos = kde_cosmos(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
ax[1].contour(X, Y, Z_cosmos, colors="C1", linewidths=1.5, label="DESI-COSMOS")

# XMM contour
kde_xmm = gaussian_kde(np.vstack([x_xmm, y_xmm]))
Z_xmm = kde_xmm(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
ax[1].contour(X, Y, Z_xmm, colors="C0", linewidths=1.5, label="DESI-XMM")
# overplot the scatter plot in the background
ax[1].scatter(x_cosmos, y_cosmos, color="C1", s=5, alpha=0.2, label="DESI-COSMOS")
ax[1].scatter(x_xmm, y_xmm, color="C0", s=5, alpha=0.2, label="DESI-XMM")
ax[1].set_xlabel("i mag")
ax[1].set_ylabel("Effective Exposure time [min]")


In [ ]:
field_names = ["HERCULES"]
fig, ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH, COLUMN_WIDTH))


cat_plot = cat[cat["FIELD_NAME"]==field_names[0]]
print(len(cat_plot))
# ax[i].grid(zorder=0)
s = ax.scatter(cat_plot["TARGET_RA"],cat_plot["TARGET_DEC"],marker=".",
            s=1,c=cat_plot["EXPTIME"]/60,vmin=15,vmax=500,cmap="viridis",rasterized=True)
ax.set_title("DESI-"+field_names[0],fontsize=BIG_SIZE)
# print(cat_plot["TARGET_DEC"].min())
# ax[i].set_aspect(1)
cbar_ax = fig.add_axes([0.95, 0.15, 0.02, 0.7])
cbar = fig.colorbar(s, fraction=0.46,cax=cbar_ax)
cbar.set_label("Exposure Time [min]",rotation=-90,labelpad = 15)


#HERCULES
ax.set_xlim(243.7, 248.5,)
ax.set_ylim( 41.5,45.1)

ax.set_xlabel("R.A. [deg]")
ax.set_ylabel("DEC. [deg]")


plt.savefig("./figs/points_on_sky_with_hercules.pdf",bbox_inches="tight",dpi=300)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(TEXT_WIDTH,0.35*TEXT_WIDTH))
i_mag =[]
i_fiber_mag = []
field_names = ["XMMLSS","COSMOS"]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    i_mag.append(cat_plot["mag_i"])
    i_fiber_mag.append(cat_plot["mag_i_fiber"])

labels = ["DESI-" + name for name in field_names]
ax[0].hist(i_mag,histtype="step",bins=30,stacked=True,color = ("white","white"))
ax[0].hist(i_mag,bins=30,color=["C0","C1"],stacked=True,rwidth=0.8, alpha=0.5,label=labels,)

ax[1].hist(i_fiber_mag,histtype="step",bins=30,stacked=True,color = ("white","white"))
ax[1].hist(i_fiber_mag,bins=30,color=["C0","C1"],stacked=True, alpha=0.5,label=labels,rwidth=0.8)


ax[0].set_ylabel("Counts")
ax[1].set_ylabel("Counts")
ax[0].set_xlabel(r"$i$-magnitude")
ax[1].set_xlabel(r"$i$-fiber-magnitude")

for i in range(2):
    L = ax[i].legend(frameon=False)
#     handles, labels = ax[i].get_legend_handles_labels()
#     handles_new = handles.copy()
    
#     for h in handles_new: h.fill=True
#     L = ax[i].legend(handles_new, labels,frameon=False)

#     for p in L.get_patches():
#         col = list(p.get_edgecolor())
#         col[-1] = 0.2
#         p.set_facecolor(col)

plt.savefig("./figs/mag_distribution.pdf",bbox_inches="tight",dpi=300)

In [ ]:
fig, ax = plt.subplots(1,2,figsize=(TEXT_WIDTH,0.35*TEXT_WIDTH))
i_mag =[]
i_fiber_mag = []
field_names = ["XMMLSS","COSMOS", "HERCULES"]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    i_mag.append(cat_plot["mag_i"])
    i_fiber_mag.append(cat_plot["mag_i_fiber"])
ax[0].hist(i_mag,histtype="step",bins=30,stacked=True,color = ("white","white","white"))
ax[0].hist(i_mag,bins=30,color=["C0","C1","C2"],stacked=True,rwidth=0.8, alpha=0.5,label=field_names,)

ax[1].hist(i_fiber_mag,histtype="step",bins=30,stacked=True,color = ("white","white","white"))
ax[1].hist(i_fiber_mag,bins=30,color=["C0","C1","C2"],stacked=True, alpha=0.5,label=field_names,rwidth=0.8)


ax[0].set_ylabel("Counts")
ax[1].set_ylabel("Counts")
ax[0].set_xlabel(r"$i$-magnitude")
ax[1].set_xlabel(r"$i$-fiber-magnitude")

for i in range(2):
    L = ax[i].legend(frameon=False)
#     handles, labels = ax[i].get_legend_handles_labels()
#     handles_new = handles.copy()
    
#     for h in handles_new: h.fill=True
#     L = ax[i].legend(handles_new, labels,frameon=False)

#     for p in L.get_patches():
#         col = list(p.get_edgecolor())
#         col[-1] = 0.2
#         p.set_facecolor(col)

plt.savefig("./figs/mag_distribution_with_hercules.pdf",bbox_inches="tight")

In [ ]:
#Three stacked histograms
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.5625*COLUMN_WIDTH))
exptime = []
field_names = ["XMMLSS","COSMOS"]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    exptime.append(cat_plot["EXPTIME"]/60)
labels = ["DESI-" + name for name in field_names]
ax.hist(exptime,bins=20,stacked=True,histtype="step",color = ("white","white"))#,range=(60,600))
ax.hist(exptime,bins=20,stacked=True,label=labels,color=["C0","C1"],alpha=0.5,rwidth=0.8)#,range=(60,600))

# ax.set_ylim(0,1100)
# ax.set_xlim(0,800)
ax.set_xlabel("Exposure Time [min]")
ax.set_ylabel("Counts")
# ax.set_yscale("log")
ax.legend(frameon=False)



# for i, bound in enumerate(times):
#     exp_sel = cat[(cat["EXPTIME"]/60 >bound[0]) & (cat["EXPTIME"]/60 <=bound[1])]["EXPTIME"]/60
#     print(np.median(exp_sel))
#     ax.axvline(bound[0], c="k",ls="--",lw=0.8,zorder=5,alpha=0.8)
#     if i!=1:
#         ax.axvline(bound[1],c="k",ls="--",lw=0.8,zorder=5,alpha=0.8)
    
#     ax.fill_betweenx(np.linspace(0.,1100),bound[0],bound[1],facecolor="k",linewidth=0,zorder=5,alpha=0.05)

# handles, labels = ax.get_legend_handles_labels()

# for h in handles_new: h.fill=True
# L = ax.legend(handles_new, labels,frameon=False)

# for p in L.get_patches():
#     col = list(p.get_edgecolor())
#     col[-1] = 0.2
#     p.set_facecolor(col)
# ax.text(-0.15,1.03,"Median:",transform=ax.transAxes)
# ax.text(0.15,1.03,"82",transform=ax.transAxes)
# ax.text(0.38,1.03,"245",transform=ax.transAxes)
# ax.text(0.53,1.03,"318",transform=ax.transAxes)
plt.savefig("./figs/exptime_distribution.pdf",bbox_inches="tight",dpi=300)

In [ ]:
#Three stacked histograms
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.5625*COLUMN_WIDTH))
exptime = []
field_names = ["XMMLSS","COSMOS", "HERCULES"]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    exptime.append(cat_plot["EXPTIME"]/60)
ax.hist(exptime,bins=20,stacked=True,histtype="step",color = ("white","white","white"))#,range=(60,600))
ax.hist(exptime,bins=20,stacked=True,label=field_names,color=["C0","C1","C2"],alpha=0.5,rwidth=0.8)#,range=(60,600))

# ax.set_ylim(0,1100)
# ax.set_xlim(0,800)
ax.set_xlabel("Exposure Time [mins]")
ax.set_ylabel("Counts")
# ax.set_yscale("log")
ax.legend(frameon=False)



# for i, bound in enumerate(times):
#     exp_sel = cat[(cat["EXPTIME"]/60 >bound[0]) & (cat["EXPTIME"]/60 <=bound[1])]["EXPTIME"]/60
#     print(np.median(exp_sel))
#     ax.axvline(bound[0], c="k",ls="--",lw=0.8,zorder=5,alpha=0.8)
#     if i!=1:
#         ax.axvline(bound[1],c="k",ls="--",lw=0.8,zorder=5,alpha=0.8)
    
#     ax.fill_betweenx(np.linspace(0.,1100),bound[0],bound[1],facecolor="k",linewidth=0,zorder=5,alpha=0.05)

# handles, labels = ax.get_legend_handles_labels()

# for h in handles_new: h.fill=True
# L = ax.legend(handles_new, labels,frameon=False)

# for p in L.get_patches():
#     col = list(p.get_edgecolor())
#     col[-1] = 0.2
#     p.set_facecolor(col)
# ax.text(-0.15,1.03,"Median:",transform=ax.transAxes)
# ax.text(0.15,1.03,"82",transform=ax.transAxes)
# ax.text(0.38,1.03,"245",transform=ax.transAxes)
# ax.text(0.53,1.03,"318",transform=ax.transAxes)
plt.savefig("./figs/exptime_distribution_with_hercules.pdf",bbox_inches="tight",dpi=300)

In [ ]:
#Without HERCULES
cat["success"] = (cat["VI_quality"]>2) & (cat["Z"]>0.01)


#Three stacked histograms
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.5625*COLUMN_WIDTH))
exptime = []
field_names = ["XMMLSS","COSMOS"]
labels = ["DESI-" + name for name in field_names]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    exptime.append(cat_plot["best_z"][cat_plot["success"]])
ax.hist(exptime,bins=50,stacked=True,histtype="step",color = ("white","white"))#,range=(60,600))
ax.hist(exptime,bins=50,stacked=True,label=labels,color=["C0","C1"],alpha=0.5,rwidth=0.8)#,range=(60,600))

ax.set_xlim(-0.1,3)
ticks = np.linspace(0,3,7)
# print(ticks)
ax.set_xticks(ticks)
# ax.set_xlim(0,800)
ax.set_xlabel("Spectroscopic Redshift")
ax.set_ylabel("Counts")
# ax.set_yscale("log")
ax.legend(frameon=False)



# cat_plot["success"] = (cat_plot["VI_quality"]>2) & (cat_plot["Z"]>0.01)

# fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.7*COLUMN_WIDTH))
# ax.hist(cat_plot["best_z"],histtype="step",bins=50)
# # ax.set_ylim(0,2)
# 
# 
fig.savefig("./figs/redshift_distribution.pdf",bbox_inches="tight")

In [ ]:
#Witht HERCULES
cat["success"] = (cat["VI_quality"]>2) & (cat["Z"]>0.01)


#Three stacked histograms
fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.5625*COLUMN_WIDTH))
exptime = []
field_names = ["XMM","COSMOS", "HERCULES"]
for i, field in enumerate(field_names):
    cat_plot = cat[cat["FIELD_NAME"]==field]
    exptime.append(cat_plot["best_z"][cat_plot["success"]])
ax.hist(exptime,bins=50,stacked=True,histtype="step",color = ("white","white","white"))#,range=(60,600))
ax.hist(exptime,bins=50,stacked=True,label=field_names,color=["C0","C1","C2"],alpha=0.5,rwidth=0.8)#,range=(60,600))

ax.set_xlim(-0.1,3)
# ax.set_xlim(0,800)
ax.set_xlim(-0.1,3)
ticks = np.linspace(0,3,7)
ax.set_xlabel("Spectroscopic Redshift")
ax.set_ylabel("Counts")
# ax.set_yscale("log")
ax.legend(frameon=False)



# cat_plot["success"] = (cat_plot["VI_quality"]>2) & (cat_plot["Z"]>0.01)

# fig, ax = plt.subplots(1,1,figsize=(COLUMN_WIDTH,0.7*COLUMN_WIDTH))
# ax.hist(cat_plot["best_z"],histtype="step",bins=50)
# # ax.set_ylim(0,2)
# 
# 
fig.savefig("./figs/redshift_distribution_with_hercules.pdf",bbox_inches="tight")

### Compare Exposure times and Fiber magnitudes

In [ ]:
data_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/MERGED")
cat = Table.read(data_path / "merged_cat_LSST_WL_Y1.fits")


In [ ]:
cat_plot = cat[np.isin(cat["FIELD_NAME"],[b"XMMLSS",b"COSMOS"])]

In [ ]:
len(cat_plot)

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH,COLUMN_WIDTH))
ax.scatter(cat_plot["EXPTIME"]/60,cat_plot["COADD_EXPTIME"]/60, marker=".",s=0.5)
x = np.linspace(0,4e4/60,100)
ax.plot(x,x,"k--")
ax.set_xlim(0,40000/60)
ax.set_ylim(0,40000/60)
ax.set_aspect("equal")
ax.set_xlabel("Effective Exposure Time [min]")
ax.set_ylabel("Observed Exposure Time [min]")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH,COLUMN_WIDTH))

ratio = cat_plot["COADD_EXPTIME"]/cat_plot["EXPTIME"]
_, bins = pd.qcut(cat_plot["EXPTIME"]/60,10,retbins=True)

med, _,_ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic="median")
p25, _, _ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic=lambda x: np.percentile(x,25))
p75,_,_ = binned_statistic(cat_plot["EXPTIME"]/60, ratio, bins=bins,statistic=lambda x: np.percentile(x,75))
better_step(bins,med,(p25,p75),ax=ax,c="C1")

ax.scatter(cat_plot["EXPTIME"]/60,ratio, marker=".",s=0.5)
ax.axhline(np.median(ratio),color="C1",ls="--",lw=1)
ax.axhline(1,c="k",ls="--")
ax.set_ylim(0.5,2.5)
ax.set_xlim(0,500)
# ax.set_aspect("equal")
# ax.set_xlabel("Effective Exposure Time [min]")
# # ax.set_ylabel("Observed Exposure Time [min]")

In [ ]:
ls_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/other_surveys/LSDR9")

In [ ]:
ls_cat = []
for file in ls_path.glob("*.csv"):
    ls_cat.append(pd.read_csv(file))
ls_cat = pd.concat(ls_cat)

In [ ]:
ls_cat

In [ ]:
def fluxToMag(flux):
    return 22.5 - 2.5 * np.log10(flux)

def mw_transmission(cat,band):
    band_coeffs = {"u":3.995, "g": 3.214, "r": 2.165, "i": 1.592, "z": 1.211, "Y": 1.064,
              "W1": 0.184,"W2": 0.113,"W3": 0.0241,"W4": 0.00910}
    A = band_coeffs[band]*cat["EBV"]
    return 10**(A/-2.5)

In [ ]:
ls_cat["rfibermag"] = fluxToMag(ls_cat["fiberflux_r"] / ls_cat["mw_transmission_r"])
ls_cat = ls_cat[np.isfinite(ls_cat["rfibermag"])]

In [ ]:
hsc_coord = SkyCoord(ra=cat_plot["ra"]*u.degree, dec = cat_plot["dec"]*u.degree)
ls_coord = SkyCoord(ra=ls_cat["ra"].values*u.degree, dec= ls_cat["dec"].values*u.degree)

In [ ]:
idx, d2d, d3d = hsc_coord.match_to_catalog_sky(ls_coord)

In [ ]:
cat_plot["rfibermag"] = ls_cat["rfibermag"].iloc[idx]
cat_plot = cat_plot[d2d.arcsec<1]

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH,COLUMN_WIDTH))
ax.scatter(cat_plot["mag_r_fiber"],cat_plot["rfibermag"], marker=".",s=0.5)
x = np.linspace(22,26)
ax.plot(x,x,"k--")
ax.set_xlim(22,26)
ax.set_ylim(22,26)
ax.set_aspect("equal")
ax.set_ylabel("LS r fiber mag")
ax.set_xlabel("HSC r fiber mag")

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(COLUMN_WIDTH,COLUMN_WIDTH))

offset = cat_plot["mag_r_fiber"] - cat_plot["rfibermag"]
_, bins = pd.qcut(cat_plot["mag_r_fiber"],10,retbins=True)

med, _,_ = binned_statistic(cat_plot["mag_r_fiber"], offset, bins=bins,statistic="median")
p25, _, _ = binned_statistic(cat_plot["mag_r_fiber"], offset, bins=bins,statistic=lambda x: np.percentile(x,25))
p75,_,_ = binned_statistic(cat_plot["mag_r_fiber"], offset, bins=bins,statistic=lambda x: np.percentile(x,75))
better_step(bins,med,(p25,p75),ax=ax,c="C1")

ax.scatter(cat_plot["mag_r_fiber"],offset, marker=".",s=0.5)
ax.axhline(np.median(offset),color="C1",ls="--",lw=1)
ax.axhline(0,c="k",ls="--")
# ax.set_ylim(0.5,2.5)
ax.set_xlim(22,26)
# ax.set_aspect("equal")
# ax.set_xlabel("Effective Exposure Time [min]")
# # ax.set_ylabel("Observed Exposure Time [min]")

In [ ]:
np.mean(offset)